# 하이브리드 추천 시스템 구현

사용자-아이템 특성을 활용한 하이브리드 추천 시스템을 구현합니다.
1. 협업 필터링 기반 추천
2. 콘텐츠 기반 필터링
3. 하이브리드 추천

In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import coo_matrix
from implicit.als import AlternatingLeastSquares
from sklearn.model_selection import train_test_split
from surprise import Reader, Dataset

In [36]:
# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

# 데이터 로드
vod_mart_data = pd.read_csv('../data/processed/vod_mart_processed.csv')
combined_data = pd.read_csv('../data/processed/combined_df.csv')

C:\Users\user\AppData\Local\Temp\ipykernel_5800\2496081781.py:6: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  vod_mart_data = pd.read_csv('../data/processed/vod_mart_processed.csv')


In [38]:
combined_data.dropna(subset = ['category'], inplace = True)

In [39]:
# 1. 데이터 전처리
def preprocess_data(df, vod_data):
    # 필요한 컬럼만 선택
    df = df.merge(
        vod_data[['asset_id', 'genre', 'category_l1', 'category_l2', 'super_asset_nm']],
        on='asset_id',
        how='left'
    )
    
    # 시청 완료율 계산
    df['completion_rate'] = df['use_tms'] / df['disp_rtm']
    df['completion_rate'] = df['completion_rate'].clip(0, 1)
    
    # 시청 시간 정규화
    scaler = MinMaxScaler()
    df['normalized_time'] = scaler.fit_transform(df[['use_tms']])
    
    return df

processed_df = preprocess_data(combined_data, vod_mart_data)

In [43]:
processed_df.dropna(subset = ['completion_rate'], inplace = True)

In [44]:
# 2. 사용자 특성 추출

def extract_user_features(df: pd.DataFrame) -> pd.DataFrame:
    # 1) 기본 통계
    user_features = df.groupby('sha2_hash').agg(
        total_sessions=('asset_id', 'count'),
        avg_watch_time=('use_tms', 'mean'),
        avg_completion_rate=('completion_rate', 'mean'),
        unique_contents=('asset_id', 'nunique')
    ).reset_index()

    # 2) 선호 장르
    genre_dummies = pd.get_dummies(df['genre'], prefix='genre')
    genre_preferences = pd.concat([df[['sha2_hash']], genre_dummies], axis=1) \
                          .groupby('sha2_hash').sum()

    # 3) 시간대별 시청 패턴
    hour_dummies = pd.get_dummies(df['hour'], prefix='hour')
    time_patterns = pd.concat([df[['sha2_hash']], hour_dummies], axis=1) \
                      .groupby('sha2_hash').mean()

    # 4) 요일별 시청 패턴
    weekday_dummies = pd.get_dummies(df['weekday_kr'], prefix='weekday')
    weekday_patterns = pd.concat([df[['sha2_hash']], weekday_dummies], axis=1) \
                         .groupby('sha2_hash').mean()

    # 5) 모든 특성 결합
    user_features = (
        user_features
        .merge(genre_preferences, on='sha2_hash')
        .merge(time_patterns,     on='sha2_hash')
        .merge(weekday_patterns,  on='sha2_hash')
    )

    return user_features

# 사용 예시
user_features = extract_user_features(processed_df)

In [45]:
def extract_item_features(df: pd.DataFrame) -> pd.DataFrame:
    # 1) 기본 콘텐츠 통계
    item_features = df.groupby('asset_id').agg(
        total_views=('sha2_hash', 'count'),
        unique_viewers=('sha2_hash', 'nunique'),
        avg_completion=('completion_rate', 'mean'),
        avg_watch_time=('use_tms', 'mean')
    ).reset_index()

    # 2) 장르 및 카테고리 원핫 인코딩
    genre_dummies = pd.get_dummies(df['genre'],      prefix='genre')
    cat1_dummies  = pd.get_dummies(df['category_l1'], prefix='cat1')
    cat2_dummies  = pd.get_dummies(df['category_l2'], prefix='cat2')
    content_metadata = pd.concat(
        [df[['asset_id']], genre_dummies, cat1_dummies, cat2_dummies],
        axis=1
    ).groupby('asset_id').first()

    # 3) 시간대별 시청 패턴
    hour_dummies  = pd.get_dummies(df['hour'], prefix='hour')
    time_patterns = pd.concat([df[['asset_id']], hour_dummies], axis=1) \
                      .groupby('asset_id').mean()

    # 4) 요일별 시청 패턴
    weekday_dummies  = pd.get_dummies(df['weekday_kr'], prefix='weekday')
    weekday_patterns = pd.concat([df[['asset_id']], weekday_dummies], axis=1) \
                           .groupby('asset_id').mean()

    # 5) 모든 특성 결합
    item_features = (
        item_features
        .merge(content_metadata, on='asset_id')
        .merge(time_patterns,     on='asset_id')
        .merge(weekday_patterns,  on='asset_id')
    )

    return item_features

# 사용 예시
item_features = extract_item_features(processed_df)

ALS 모델 학습

In [46]:
# 1) 인덱스 맵핑: 문자열 → 정수
users, user_idx = np.unique(processed_df['sha2_hash'], return_inverse=True)
items, item_idx = np.unique(processed_df['asset_id'], return_inverse=True)

# 2) 희소행렬 생성 (items × users)
data   = processed_df['completion_rate'].values
mat = coo_matrix((data, (item_idx, user_idx)))

# 3) ALS 모델 초기화 & 학습
als_model = AlternatingLeastSquares(
    factors=100, regularization=0.01, iterations=20, calculate_training_loss = True,use_gpu=False
)
# implicit 라이브러리는 (item×user) confidence 행렬을 기대
als_model.fit(mat)
# 4) 잠재요인 저장
user_factors = als_model.user_factors       # shape=(n_users, factors)
item_factors = als_model.item_factors       # shape=(n_items, factors)

c:\Users\user\anaconda3\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 1.1653828620910645 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

In [65]:
# 콘텐츠 기반 CB 연산에 필요한 행렬·노름 초기화
X      = item_features.drop('asset_id', axis=1).astype(np.float64).to_numpy()  # (n_items, n_feats)
norms  = np.linalg.norm(X, axis=1)                      # (n_items,)
X_norm = X / norms[:, None]                             # 사전 정규화

In [78]:
# 인덱스 맵핑 dict 생성 (O(1) 조회용)
user2idx = {u:i for i,u in enumerate(users)}
item2idx = { item: idx for idx, item in enumerate(items) }

In [79]:
items

array(['CCS00000000000000101', 'CCS00000000000000601',
       'CCS00000000000001101', ..., 'M5194025LSGM62417001',
       'M5194026LSGM62417901', 'M5194027LSGM62418801'], dtype=object)

In [80]:
class HybridImplicitRecommender:
    def __init__(self, cf_weight=0.7, cb_weight=0.3, first_stage_size=1000):
        self.cf_weight       = cf_weight
        self.cb_weight       = cb_weight
        self.first_stage_size = first_stage_size

    def recommend(self, user_id, N=10, exclude_seen=True):
        # 1) 사용자 인덱스
        u = user2idx.get(user_id)
        if u is None:
            return []  # 신규 사용자

        # 2) 후보 아이템 + 인덱스
        seen      = set(processed_df.loc[processed_df['sha2_hash']==user_id, 'asset_id'])
        candidates= [it for it in items if (not exclude_seen) or (it not in seen)]
        cidx_all  = np.array([item2idx[it] for it in candidates])

        # 3) CF 점수 전체 계산
        user_vec     = user_factors[u]                  # (factors,)
        cf_scores_all= item_factors[cidx_all] @ user_vec  # (n_cand,)

        # 4) 1차 랭킹: CF 기준 상위 K1
        K1   = min(self.first_stage_size, len(cidx_all))
        idx1 = np.argpartition(cf_scores_all, -K1)[-K1:]  # CF 점수 상위 K1(unsorted)
        cidx1        = cidx_all[idx1]                    # 실제 아이템 인덱스
        cand1        = [candidates[i] for i in idx1]
        cf1          = cf_scores_all[idx1]

        # 5) 사용자 프로파일 벡터 (가중합)
        watched = processed_df.loc[processed_df['sha2_hash']==user_id, 'asset_id']
        widxs   = np.array([item2idx[it] for it in watched])
        weights = processed_df.loc[processed_df['sha2_hash']==user_id, 'completion_rate'].values

        W         = X[widxs]                          # (n_watched, n_feats)
        user_prof = (weights.reshape(1,-1) @ W).flatten()  # (n_feats,)
        up_norm   = np.linalg.norm(user_prof) or 1.0
        user_prof /= up_norm                         # 단위 벡터화

        # 6) CB 점수 (1차 후보만)
        # X_norm[cidx1]: 이미 노름으로 나눠진 후보 피처
        cb1 = X_norm[cidx1] @ user_prof              # (K1,)

        # 7) 정규화(normalize) & 가중합
        cf_n = cf1 / (cf1.max() or 1)
        cb_n = cb1 / (cb1.max() or 1)
        final= self.cf_weight * cf_n + self.cb_weight * cb_n

        # 8) 최종 Top-N
        topk = np.argsort(final)[-N:][::-1]
        return [(cand1[i], final[i]) for i in topk]

In [81]:
recommender = HybridImplicitRecommender(cf_weight=0.7, cb_weight=0.3, first_stage_size=1000)

sample_user = processed_df['sha2_hash'].iloc[1]
top10 = recommender.recommend(sample_user, N=10)
for asset_id, score in top10:
    info = vod_mart_data[vod_mart_data['asset_id']==asset_id].iloc[0]
    print(f"{info['asset_nm'][:50]}... ({info['genre']}) — 점수: {score:.3f}")

(SD)(명작)무인시대 113회... (드라마) — 점수: 0.745
환희의 찬가2 in 상하이 24회(23/07/24)... (드라마) — 점수: 0.732
전원일기 541회... (드라마) — 점수: 0.687
핑크퐁 홈스쿨 생활습관(영어) 02회... (학습) — 점수: 0.673
백옥사무하: 내 사랑 망나니 27회.... (외화 시리즈) — 점수: 0.644
12주의 기적 13회... (기타) — 점수: 0.640
디어 마이 프렌즈 01회... (드라마) — 점수: 0.595
VIP 10회... (드라마) — 점수: 0.593
뽕숭아학당 47회... (연예오락) — 점수: 0.585
진짜가 나타났다! 20회(23/05/28)... (드라마) — 점수: 0.535


In [82]:
recommender = HybridImplicitRecommender(cf_weight=0.7, cb_weight=0.3)

# 5. 추천 결과 생성 및 평가
def evaluate_recommendations(user_id, n=10):
    # HybridImplicitRecommender의 recommend 메소드를 사용합니다.
    recommendations = recommender.recommend(user_id, N=n)
    
    print(f"=== 사용자 {user_id[:8]}... 에 대한 추천 결과 ===")
    for item_id, score in recommendations:
        # vod_mart_data에서 asset_id로 메타정보 조회
        content_info = vod_mart_data.loc[
            vod_mart_data['asset_id'] == item_id
        ].iloc[0]
        
        print(f"제목: {content_info['asset_nm'][:50]}...")
        print(f"장르: {content_info['genre']}")
        print(f"추천 점수: {score:.3f}\n")

# 샘플 사용자에 대한 추천 평가
sample_user = processed_df['sha2_hash'].iloc[0]
evaluate_recommendations(sample_user, n=10)


=== 사용자 7b59d339... 에 대한 추천 결과 ===
제목: 브레드이발소2. 06회...
장르: 애니메이션
추천 점수: 0.771

제목: 선을 넘는 녀석들 리턴즈 42회...
장르: 연예/오락
추천 점수: 0.738

제목: 중명위-대명기밀 12회....
장르: 외화 시리즈
추천 점수: 0.702

제목: (더빙)돼지저금통 part 4 13회(G)...
장르: 기타
추천 점수: 0.682

제목: (더빙)돼지저금통 Part 11 02회...
장르: 명랑/코믹
추천 점수: 0.673

제목: 타짜 17회...
장르: 미니시리즈
추천 점수: 0.670

제목: 최강 전사 미니특공대 01회...
장르: 애니메이션
추천 점수: 0.639

제목: TV동물농장-우린 같이 산다 10회(23/03/02)...
장르: 시사/교양
추천 점수: 0.582

제목: 핑크퐁 몬스터 트럭 (영어) 01회...
장르: 학습
추천 점수: 0.580

제목: (더빙)자몽TV 네버엔딩 시즌4 10회(G)...
장르: 기타
추천 점수: 0.544



In [83]:
processed_df[processed_df['sha2_hash'] == sample_user]

,sha2_hash,asset,asset_nm,CT_CL,genre_of_ct_cl,use_tms,disp_rtm,strt_dt,category,strt_dt_dt,...,asset_id,use_tms_1,watch_complete,rate,genre,category_l1,category_l2,super_asset_nm,completion_rate,normalized_time
0,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M5047991LFOJ44245901,전국민민원해결프로젝트-일꾼의탄생 66회(23/04/12),TV 시사/교양,기타,2880,2880,20230503223806,KBS/(HD)KBS 시사교양/전국민민원해결프로젝트-일꾼의탄생,2023-05-03 22:38:06,...,M5047991LFOJ44245901,2880,1,0,시사/교양,KBS,(HD)KBS 시사교양,전국민민원해결프로젝트-일꾼의탄생,1.000000,0.033515
40,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M5047991LFOJ44245901,전국민민원해결프로젝트-일꾼의탄생 66회(23/04/12),TV 시사/교양,기타,50,2880,20230503214620,KBS/(HD)KBS 시사교양/전국민민원해결프로젝트-일꾼의탄생,2023-05-03 21:46:20,...,M5047991LFOJ44245901,50,0,0,시사/교양,KBS,(HD)KBS 시사교양,전국민민원해결프로젝트-일꾼의탄생,0.017361,0.000582
68819,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M4826549LFOL80567701,조선주먹,영화,액션/어드벤쳐,484,5460,20230517005329,영화/무료영화/액션,2023-05-17 00:53:29,...,M4826549LFOL80567701,484,0,15,액션/어드벤쳐,영화,무료영화,조선주먹,0.088645,0.005632
68923,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M4826549LFOL80567701,조선주먹,영화,액션/어드벤쳐,5383,5460,20230517010331,영화/무료영화/액션,2023-05-17 01:03:31,...,M4826549LFOL80567701,5383,0,15,액션/어드벤쳐,영화,무료영화,조선주먹,0.985897,0.062643
70204,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M4826549LFOL80567701,조선주먹,영화,액션/어드벤쳐,317,5460,20230526023327,영화/무료영화/액션,2023-05-26 02:33:27,...,M4826549LFOL80567701,317,0,15,액션/어드벤쳐,영화,무료영화,조선주먹,0.058059,0.003689
71135,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M4826549LFOL80567701,조선주먹,영화,액션/어드벤쳐,2701,5460,20230523013411,영화/무료영화/액션,2023-05-23 01:34:11,...,M4826549LFOL80567701,2701,0,15,액션/어드벤쳐,영화,무료영화,조선주먹,0.494689,0.031432
163948,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M5009802LFOL98638401,(FREE)전설의 검(2020)(무료),영화,액션/어드벤쳐,197,4380,20230523012947,프리미엄 무료관/무료영화/무료 영화관,2023-05-23 01:29:47,...,M5009802LFOL98638401,197,0,15,액션/어드벤쳐,프리미엄 무료관,무료영화,전설의 검(2020),0.044977,0.002293
344120,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M4860383LFOL98636901,(FREE)#살아있다(무료),영화,액션/어드벤쳐,692,5880,20230526014854,프리미엄 무료관/무료영화/무료 영화관,2023-05-26 01:48:54,...,M4860383LFOL98636901,692,0,15,액션/어드벤쳐,프리미엄 무료관,무료영화,#살아있다,0.117687,0.008053
390384,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M0463068LFOL98637301,(FREE)기술자들(무료),영화,액션/어드벤쳐,692,7020,20230526020113,프리미엄 무료관/무료영화/무료 영화관,2023-05-26 02:01:13,...,M0463068LFOL98637301,692,0,15,액션/어드벤쳐,프리미엄 무료관,무료영화,기술자들,0.098575,0.008053
650015,7b59d3397a5ed75dbb6d2938e409f412cf03bc0ca86dbc...,cjc|M4864275LFOL85031701,정무문2: 전설의 흑협,영화,액션/어드벤쳐,534,4560,20230517023402,영화/무료영화/액션,2023-05-17 02:34:02,...,M4864275LFOL85031701,534,0,15,액션/어드벤쳐,영화,무료영화,정무문2: 전설의 흑협,0.117105,0.006214
